# Vígil.ia — Melhorar a qualidade das bounding boxes (YOLO11x · A100)

Foca no **campeão YOLO11x**. Implementa o plano `melhorar_boxes.md` na ordem:

1. **Etapa 1 — inferência (grátis):** varredura imgsz × iou com saída visual → melhor config SEM retreinar
2. **Etapa 2 — auditoria de anotações:** box frouxa na inferência quase sempre é box frouxa no label
3. **Etapa 3 — retreino focado em localização** (só se 1–2 não bastarem): YOLO11x com `box=10`
4. **Etapa 4 — filtros de pós-processamento** (paralelo): filtro de área + ROI

⛔ **Não pule pra Etapa 3 antes de esgotar 1 e 2** (regra do plano).

**Autossuficiente:** reconstrói o dataset v3 sozinho se a sessão for nova.
Pré-requisitos no Drive: `soja_yolo11x_v3.pt`, `teste_soja.mp4`, fotos reais.
Opcional (p/ Etapa 3 isolada): `soja_yolo11x_base.pt`.

## 0. Setup e caminhos

In [ ]:
!pip -q install "ultralytics==8.4.80"

import torch, ultralytics
ultralytics.checks()
assert torch.cuda.is_available(), 'Sem GPU!'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import os, glob
from google.colab import drive
drive.mount('/content/drive')

YOLOX_PT = '/content/drive/MyDrive/soja_yolo11x_v3.pt'
assert os.path.exists(YOLOX_PT), 'soja_yolo11x_v3.pt não encontrado no Drive!'

VIDEO_TESTE = '/content/drive/MyDrive/teste_soja.mp4'

REAL_SRCS = [
    '/content/drive/MyDrive/Soja total/Soja total/Lotes',
    '/content/drive/MyDrive/Soja pra completar',
]

NAMES = ['broken', 'immature', 'intact', 'skin-damaged', 'spotted']

# Se você AINDA estiver na sessão do tira-teima, salva o estágio base do 11x
# (deixa a Etapa 3 mais isolada). Se não estiver, sem problema.
_base_src = '/content/runs/detect/runs_cmp/yolo11x_base/weights/best.pt'
if os.path.exists(_base_src) and not os.path.exists('/content/drive/MyDrive/soja_yolo11x_base.pt'):
    !cp {_base_src} /content/drive/MyDrive/soja_yolo11x_base.pt
    print('backup do estágio base feito: soja_yolo11x_base.pt')

## 1. Dataset v3 — reconstrói se a sessão for nova
Mesmas funções do `melhoria_rtdetr_v3.ipynb` (balanceamento + blur + multi-grão).
Necessário para as Etapas 2 e 3. Se `/content/soja_det_v3` já existir, reusa.

In [ ]:
import glob, hashlib, unicodedata, cv2, yaml
import numpy as np

ALIASES = {0: ['broken', 'quebrad'], 1: ['immature', 'imatur', 'nao maduro'],
           2: ['intact'], 3: ['skin', 'casca', 'ardid', 'danific'], 4: ['spotted', 'manchad']}
IGNORE = ['part of the original']
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
RNG = np.random.default_rng(42)

def norm(s):
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode().lower()

def class_of(folder):
    n = norm(folder)
    if any(norm(k) in n for k in IGNORE):
        return None
    for idx in range(5):
        if any(norm(k) in n for k in ALIASES[idx]):
            return idx
    return None

def collect_real(srcs, val_frac=0.15):
    items = []
    for src in srcs:
        for root, _, files in os.walk(src):
            cls = None
            for part in reversed(root.split(os.sep)):
                c = class_of(part)
                if c is not None:
                    cls = c; break
            if cls is None:
                continue
            for fn in files:
                if fn.lower().endswith(IMG_EXT):
                    p = os.path.join(root, fn)
                    h = int(hashlib.md5(p.encode()).hexdigest(), 16)
                    items.append((p, cls, 'val' if (h % 100) < val_frac * 100 else 'train'))
    from collections import Counter
    print('coletado:', dict(Counter(sp for _, _, sp in items)))
    return items

def sat_box(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    s = cv2.GaussianBlur(hsv[:, :, 1], (5, 5), 0)
    _, th = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    h, w = img.shape[:2]
    if area < 0.01 * h * w or area > 0.90 * h * w:
        return None
    x, y, bw, bh = cv2.boundingRect(c)
    pad = int(0.04 * min(bw, bh)) + 2
    x1, y1 = max(0, x - pad), max(0, y - pad)
    x2, y2 = min(w, x + bw + pad), min(h, y + bh + pad)
    return (((x1 + x2) / 2) / w, ((y1 + y2) / 2) / h, (x2 - x1) / w, (y2 - y1) / h)

def otsu_box(img):
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    if area < 0.005 * h * w or area > 0.995 * h * w:
        return None
    x, y, bw, bh = cv2.boundingRect(c)
    pad = int(0.04 * min(bw, bh)) + 2
    x1, y1 = max(0, x - pad), max(0, y - pad)
    x2, y2 = min(w, x + bw + pad), min(h, y + bh + pad)
    return (((x1 + x2) / 2) / w, ((y1 + y2) / 2) / h, (x2 - x1) / w, (y2 - y1) / h)

def letterbox640(img, size=640):
    h, w = img.shape[:2]
    s = size / max(h, w)
    img = cv2.resize(img, (max(1, round(w * s)), max(1, round(h * s))))
    h, w = img.shape[:2]
    top, left = (size - h) // 2, (size - w) // 2
    img = cv2.copyMakeBorder(img, top, size - h - top, left, size - w - left,
                             cv2.BORDER_CONSTANT, value=(0, 0, 0))
    return img, s, left, top

def motion_blur(img, rng=RNG):
    k = int(rng.choice([7, 9, 11, 13, 15]))
    kernel = np.zeros((k, k), np.float32)
    kernel[k // 2, :] = 1.0
    M = cv2.getRotationMatrix2D((k / 2 - 0.5, k / 2 - 0.5), float(rng.uniform(0, 180)), 1)
    kernel = cv2.warpAffine(kernel, M, (k, k))
    kernel /= max(kernel.sum(), 1e-6)
    return cv2.filter2D(img, -1, kernel)

def extract_cutout(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    s = cv2.GaussianBlur(hsv[:, :, 1], (5, 5), 0)
    _, th = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    h, w = img.shape[:2]
    if area < 0.01 * h * w or area > 0.90 * h * w:
        return None
    mask = np.zeros((h, w), np.uint8)
    cv2.drawContours(mask, [c], -1, 255, -1)
    x, y, bw, bh = cv2.boundingRect(c)
    return img[y:y + bh, x:x + bw], mask[y:y + bh, x:x + bw]

def make_scene(cutouts, rng=RNG, size=640):
    bg = int(rng.integers(20, 130))
    canvas = np.clip(np.full((size, size, 3), bg, np.int16)
                     + rng.normal(0, 6, (size, size, 3)), 0, 255).astype(np.uint8)
    occ = np.zeros((size, size), np.uint8)
    boxes = []
    for _ in range(int(rng.integers(6, 26))):
        cls, crop, mask = cutouts[int(rng.integers(len(cutouts)))]
        s = int(rng.integers(60, 150)) / max(crop.shape[:2])
        crop2 = cv2.resize(crop, None, fx=s, fy=s)
        mask2 = cv2.resize(mask, None, fx=s, fy=s, interpolation=cv2.INTER_NEAREST)
        h2, w2 = crop2.shape[:2]
        diag = int(np.ceil(np.hypot(h2, w2))) + 2
        M = cv2.getRotationMatrix2D((w2 / 2, h2 / 2), float(rng.uniform(0, 360)), 1)
        M[0, 2] += (diag - w2) / 2
        M[1, 2] += (diag - h2) / 2
        crop3 = cv2.warpAffine(crop2, M, (diag, diag))
        mask3 = cv2.warpAffine(mask2, M, (diag, diag), flags=cv2.INTER_NEAREST)
        ys, xs = np.where(mask3 > 0)
        if not len(xs):
            continue
        crop3 = crop3[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
        mask3 = mask3[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
        gh, gw = mask3.shape
        if gh >= size - 2 or gw >= size - 2:
            continue
        placed = False
        for _try in range(20):
            px = int(rng.integers(0, size - gw))
            py = int(rng.integers(0, size - gh))
            inter = (occ[py:py + gh, px:px + gw] > 0) & (mask3 > 0)
            if inter.sum() <= 0.15 * (mask3 > 0).sum():
                placed = True
                break
        if not placed:
            continue
        alpha = (cv2.GaussianBlur(mask3, (5, 5), 0).astype(np.float32) / 255)[..., None]
        reg = canvas[py:py + gh, px:px + gw]
        canvas[py:py + gh, px:px + gw] = (alpha * crop3 + (1 - alpha) * reg).astype(np.uint8)
        occ[py:py + gh, px:px + gw][mask3 > 0] = 255
        boxes.append((cls, (px + gw / 2) / size, (py + gh / 2) / size, gw / size, gh / size))
    return canvas, boxes

def balance_train(items):
    from collections import defaultdict
    train = [it for it in items if it[2] == 'train']
    rest = [it for it in items if it[2] != 'train']
    by = defaultdict(list)
    for it in train:
        by[it[1]].append(it)
    mx = max(len(v) for v in by.values())
    out = []
    for c, v in by.items():
        out += v + [v[int(i)] for i in RNG.integers(0, len(v), mx - len(v))]
    print('balanceado (train):', {NAMES[c]: sum(1 for it in out if it[1] == c) for c in sorted(by)})
    return out + rest

def build_v3(items, out_dir, n_synth=600, blur_frac=0.4):
    assert items, 'Nenhuma imagem coletada! Confira REAL_SRCS.'
    for sp in ('train', 'val', 'test'):
        os.makedirs(f'{out_dir}/images/{sp}', exist_ok=True)
        os.makedirs(f'{out_dir}/labels/{sp}', exist_ok=True)
    items = balance_train(items)
    cutouts = []
    kept = skipped = 0
    for i, (path, cls, sp) in enumerate(items):
        if i % 200 == 0:
            print(f'  fotos {i}/{len(items)}…', flush=True)
        img = cv2.imread(path)
        if img is None:
            skipped += 1; continue
        h0, w0 = img.shape[:2]
        box = sat_box(img) or otsu_box(img)
        if box is None:
            skipped += 1; continue
        if sp == 'train':
            cut = extract_cutout(img)
            if cut is not None:
                cutouts.append((cls, cut[0], cut[1]))
        lb, s, left, top = letterbox640(img)
        cx, cy, ww, hh = box
        cx = (cx * w0 * s + left) / 640.0
        cy = (cy * h0 * s + top) / 640.0
        ww = (ww * w0 * s) / 640.0
        hh = (hh * h0 * s) / 640.0
        line = f'{cls} {cx:.6f} {cy:.6f} {ww:.6f} {hh:.6f}'
        stem = f'{sp}_{i:06d}'
        cv2.imwrite(f'{out_dir}/images/{sp}/{stem}.jpg', lb, [cv2.IMWRITE_JPEG_QUALITY, 95])
        open(f'{out_dir}/labels/{sp}/{stem}.txt', 'w').write(line)
        kept += 1
        if sp == 'train':
            cv2.imwrite(f'{out_dir}/images/train/{stem}b.jpg', motion_blur(lb),
                        [cv2.IMWRITE_JPEG_QUALITY, 95])
            open(f'{out_dir}/labels/train/{stem}b.txt', 'w').write(line)
            kept += 1
    print(f'fotos reais: kept={kept} skipped={skipped} | recortes: {len(cutouts)}')
    assert cutouts, 'Nenhum recorte extraído!'
    synth = 0
    for j in range(n_synth):
        if j % 100 == 0:
            print(f'  cenas {j}/{n_synth}…', flush=True)
        canvas, boxes = make_scene(cutouts)
        if not boxes:
            continue
        if RNG.random() < blur_frac:
            canvas = motion_blur(canvas)
        stem = f'synth_{j:05d}'
        cv2.imwrite(f'{out_dir}/images/train/{stem}.jpg', canvas, [cv2.IMWRITE_JPEG_QUALITY, 95])
        open(f'{out_dir}/labels/train/{stem}.txt', 'w').write(
            '\n'.join(f'{c} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}' for c, cx, cy, w, h in boxes))
        synth += 1
    print(f'cenas sintéticas: {synth}')
    yaml.safe_dump({'path': out_dir, 'train': 'images/train', 'val': 'images/val',
                    'test': 'images/test', 'names': {i: n for i, n in enumerate(NAMES)}},
                   open(f'{out_dir}/data.yaml', 'w'), sort_keys=False, allow_unicode=True)
    return f'{out_dir}/data.yaml'

DS = '/content/soja_det_v3'
V3_YAML = f'{DS}/data.yaml'
if not os.path.exists(V3_YAML):
    print('dataset v3 não está na sessão — construindo (uns 10–15 min)…')
    V3_YAML = build_v3(collect_real(REAL_SRCS), DS)
print('dataset:', V3_YAML)

# ETAPA 1 — Varredura de inferência (grátis, rodar primeiro)
Extrai frames de teste do vídeo e roda o YOLO11x com imgsz ∈ {640, 960, 1280}
× iou ∈ {0.45, 0.5, 0.6}. Imagens anotadas em pastas separadas + zip no Drive.

In [ ]:
FRAMES_DIR = '/content/frames_teste'
N_FRAMES = 10
os.makedirs(FRAMES_DIR, exist_ok=True)
if not glob.glob(f'{FRAMES_DIR}/*.jpg'):
    cap = cv2.VideoCapture(VIDEO_TESTE)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step = max(1, total // (N_FRAMES + 1))
    for k, idx in enumerate(range(step, total, step)):
        if k >= N_FRAMES:
            break
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()
        if ok:
            cv2.imwrite(f'{FRAMES_DIR}/frame_{k:02d}.jpg', frame)
    cap.release()
print('frames de teste:', len(glob.glob(f'{FRAMES_DIR}/*.jpg')))

In [ ]:
import shutil
from ultralytics import YOLO

model = YOLO(YOLOX_PT)
SWEEP = '/content/sweep_boxes'
shutil.rmtree(SWEEP, ignore_errors=True)

for sz in (640, 960, 1280):
    for iou in (0.45, 0.5, 0.6):
        name = f'yolo11x_imgsz{sz}_iou{iou}'
        model.predict(source=FRAMES_DIR, imgsz=sz, iou=iou, conf=0.35,
                      agnostic_nms=True, save=True, project=SWEEP,
                      name=name, exist_ok=True, verbose=False, device=0)
        print('ok:', name)

shutil.make_archive('/content/sweep_boxes_resultados', 'zip', SWEEP)
!cp /content/sweep_boxes_resultados.zip /content/drive/MyDrive/
print('\nZIP com todas as combinações no Drive: sweep_boxes_resultados.zip')

In [ ]:
# Grade comparativa de UM frame (o mesmo) em todas as 9 configs
import matplotlib.pyplot as plt

FRAME_REF = 'frame_04.jpg'   # troque pra inspecionar outro
cfgs = [(sz, iou) for sz in (640, 960, 1280) for iou in (0.45, 0.5, 0.6)]
plt.figure(figsize=(16, 14))
for i, (sz, iou) in enumerate(cfgs):
    p = f'{SWEEP}/yolo11x_imgsz{sz}_iou{iou}/{FRAME_REF}'
    if not os.path.exists(p):
        continue
    ax = plt.subplot(3, 3, i + 1)
    ax.imshow(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB))
    ax.set_title(f'imgsz={sz}  iou={iou}', fontsize=10)
    ax.axis('off')
plt.suptitle('YOLO11x — mesma cena, 9 configs (procure a box mais justa)')
plt.tight_layout(); plt.show()

### Como ler a Etapa 1
- **imgsz maior** (960/1280) tende a apertar a box de grão pequeno — mas custa
  latência (~2,2× a 960, ~4× a 1280) e o modelo foi TREINADO em 640; se ficar
  estranho em 1280, é mismatch de escala, não bug.
- **iou menor** (0.45–0.5) mata box frouxa duplicada em grão colado; se grãos
  encostados começarem a SUMIR, o iou ficou agressivo demais.
- Anote a config vencedora — ela vai pro pipeline (demo/track) e pra Etapa 4.
  **Se a box ficou justa aqui, PARE — não precisa retreinar.**

# ETAPA 2 — Auditoria das anotações (labels frouxos → boxes frouxas)

In [ ]:
import random

def load_labels(split):
    rows = []
    for lp in glob.glob(f'{DS}/labels/{split}/*.txt'):
        for ln in open(lp):
            c, cx, cy, w, h = ln.split()
            rows.append((int(c), float(cx), float(cy), float(w), float(h)))
    return rows

def audit(split):
    rows = load_labels(split)
    if not rows:
        return
    areas = np.array([w * h for _, _, _, w, h in rows])
    touch = sum(1 for _, cx, cy, w, h in rows
                if cx - w/2 < 0.004 or cy - h/2 < 0.004 or cx + w/2 > 0.996 or cy + h/2 > 0.996)
    print(f'\n=== {split}: {len(rows)} caixas ===')
    print(f'  área (fração do frame): mediana={np.median(areas):.3f}  '
          f'p5={np.percentile(areas,5):.4f}  p95={np.percentile(areas,95):.3f}')
    print(f'  GIGANTES (>30% do frame): {int((areas > 0.30).sum())} '
          f'({100*(areas > 0.30).mean():.1f}%)  <- suspeitas de label frouxo/errado')
    print(f'  minúsculas (<0.2% do frame): {int((areas < 0.002).sum())}')
    print(f'  tocando a borda: {touch} ({100*touch/len(rows):.1f}%)')
    per = {}
    for c, _, _, w, h in rows:
        per.setdefault(c, []).append(w * h)
    for c in sorted(per):
        print(f'  {NAMES[c]:14s} n={len(per[c]):5d}  área mediana={np.median(per[c]):.3f}')
    plt.figure(figsize=(6, 3))
    plt.hist(areas, bins=60)
    plt.title(f'{split} — distribuição da área da caixa (fração do frame)')
    plt.tight_layout(); plt.show()

for sp in ('train', 'val'):
    audit(sp)

In [ ]:
# Inspeção visual: amostras aleatórias com ground-truth. Rode várias vezes.
def show_gt(prefix, title, n=6):
    paths = [p for p in glob.glob(f'{DS}/images/train/*.jpg')
             if os.path.basename(p).startswith(prefix)]
    random.shuffle(paths)
    plt.figure(figsize=(13, 8))
    for i, p in enumerate(paths[:n]):
        img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        for ln in open(p.replace('/images/', '/labels/').replace('.jpg', '.txt')):
            c, cx, cy, ww, hh = ln.split()
            cx, cy, ww, hh = float(cx), float(cy), float(ww), float(hh)
            x1, y1 = int((cx - ww/2) * w), int((cy - hh/2) * h)
            cv2.rectangle(img, (x1, y1), (int((cx + ww/2) * w), int((cy + hh/2) * h)), (0, 255, 0), 2)
            cv2.putText(img, NAMES[int(c)], (x1, max(14, y1 - 4)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 0), 1)
        ax = plt.subplot(2, 3, i + 1); ax.imshow(img); ax.axis('off')
    plt.suptitle(title); plt.tight_layout(); plt.show()

show_gt('train_', 'FOTOS REAIS — o label abraça o grão?')
show_gt('synth_', 'CENAS SINTÉTICAS — caixas justas nos grãos colados?')

### Como ler a Etapa 2
- Labels frouxos nas FOTOS REAIS → problema no pseudo-rótulo (padding do `sat_box`
  é ~4% + 2px; dá pra reduzir e reconstruir) — me avise que ajusto o builder.
- Muitas caixas >30% do frame → label errado (o bug da "caixa gigante") — me mande o print.
- Labels OK mas inferência frouxa na melhor config da Etapa 1 → aí sim a Etapa 3 se justifica.

# ETAPA 3 — Retreino focado em localização (SÓ depois de 1–2)
Repete o fine-tune v3 mudando APENAS `box` (peso da loss de localização) —
mesmo ponto de partida, mesmos dados/épocas → efeito isolado. `box=10` (padrão 7.5).
Batch 16 calibrado p/ A100 40GB (~22 GB); OOM auto-retry cobre imprevistos.

In [ ]:
# ponto de partida: estágio base do 11x (isola o efeito); fallback = campeão v3
YOLOX_BASE = '/content/runs/detect/runs_cmp/yolo11x_base/weights/best.pt'
if not os.path.exists(YOLOX_BASE):
    YOLOX_BASE = '/content/drive/MyDrive/soja_yolo11x_base.pt'
if not os.path.exists(YOLOX_BASE):
    YOLOX_BASE = YOLOX_PT
    print('⚠️ estágio base do 11x não encontrado — partindo do campeão v3.')
    print('   (comparável, mas menos isolado; lembre do caso ft_v3_disc)')
print('partindo de:', YOLOX_BASE)

YOLO(YOLOX_BASE).train(
    data=V3_YAML, epochs=60, imgsz=640, batch=16, device=0, seed=42,
    optimizer='AdamW', lr0=0.001, patience=20,
    box=10.0,            # <- ÚNICA mudança de receita (padrão 7.5)
    close_mosaic=10,
    cache=False, workers=8,
    mosaic=1.0, hsv_v=0.5, degrees=15, translate=0.1, scale=0.5,
    fliplr=0.5, flipud=0.5,
    project='runs_box', name='yolo11x_v3_box10', exist_ok=True,
)
!cp /content/runs/detect/runs_box/yolo11x_v3_box10/weights/best.pt /content/drive/MyDrive/soja_yolo11x_v3_box10.pt
print('backup ok: soja_yolo11x_v3_box10.pt')

In [ ]:
# A/B da Etapa 3: mAP50-95 (a métrica de box justa) antes vs depois
print('mAP50-95 = quão MILIMETRICAMENTE justa a caixa é (a métrica desta task)\n')
for tag, w in [('yolo11x v3 (atual)', YOLOX_PT),
               ('yolo11x box=10', '/content/runs/detect/runs_box/yolo11x_v3_box10/weights/best.pt')]:
    r = YOLO(w).val(data=V3_YAML, split='val', imgsz=640, device=0, verbose=False)
    print(f'{tag:22s} mAP50={r.box.map50:.3f}  mAP50-95={r.box.map:.3f}')

# ETAPA 4 — Filtros de pós-processamento (paralelo ao treino)
- **Filtro de área:** mata a caixa gigante da borda da mesa/transição de fundo.
- **ROI:** infere só na região da bandeja (ignora fundo, caderno, sujeira).

In [ ]:
# ---- funções de filtro (copiar pro pipeline final) ----
MAX_AREA_FRAC = 0.15              # caixa > 15% do frame = lixo (ajuste ao cenário)
ROI = (0.05, 0.05, 0.95, 0.95)   # (x1,y1,x2,y2) fração do frame; None = desligado

def filtra_area(xyxy, frame_w, frame_h, max_frac=MAX_AREA_FRAC):
    x1, y1, x2, y2 = xyxy
    return (x2 - x1) * (y2 - y1) <= max_frac * frame_w * frame_h

def recorta_roi(frame, roi=ROI):
    if roi is None:
        return frame, 0, 0
    h, w = frame.shape[:2]
    x1, y1 = int(roi[0] * w), int(roi[1] * h)
    x2, y2 = int(roi[2] * w), int(roi[3] * h)
    return frame[y1:y2, x1:x2], x1, y1

# visualiza a ROI no 1º frame pra calibrar
cap = cv2.VideoCapture(VIDEO_TESTE)
ok, fr = cap.read()
cap.release()
h, w = fr.shape[:2]
vis = fr.copy()
cv2.rectangle(vis, (int(ROI[0]*w), int(ROI[1]*h)), (int(ROI[2]*w), int(ROI[3]*h)), (0, 255, 255), 3)
plt.figure(figsize=(8, 5))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title('ROI atual (amarelo) — ajuste a tupla ROI até cobrir só a bandeja')
plt.axis('off'); plt.show()

In [ ]:
# Demonstração: vídeo com veredito travado + ROI + filtro de área
from collections import defaultdict, Counter

BEST_IMGSZ = 640   # <- coloque a config vencedora da Etapa 1
BEST_IOU = 0.5
LOCK_MIN_FRAMES, LOCK_RATIO = 8, 0.60
COLORS = {'intact': (80, 200, 80), 'immature': (60, 200, 200),
          'broken': (200, 100, 160), 'skin-damaged': (60, 160, 255),
          'spotted': (80, 80, 230)}

votes, seen, locked = defaultdict(Counter), Counter(), {}
cap = cv2.VideoCapture(VIDEO_TESTE)
fps = cap.get(cv2.CAP_PROP_FPS) or 30
writer = None
OUT = '/content/video_boxes_filtrado.mp4'

while True:
    ok, frame = cap.read()
    if not ok:
        break
    if writer is None:
        H, W = frame.shape[:2]
        writer = cv2.VideoWriter(OUT, cv2.VideoWriter_fourcc(*'mp4v'), fps, (W, H))
    crop, ox, oy = recorta_roi(frame)
    r = model.track(crop, imgsz=BEST_IMGSZ, iou=BEST_IOU, conf=0.35,
                    agnostic_nms=True, tracker='bytetrack.yaml',
                    persist=True, verbose=False)[0]
    if r.boxes.id is not None:
        for xyxy, tid, c, cf in zip(r.boxes.xyxy.cpu().numpy(),
                                    r.boxes.id.int().tolist(),
                                    r.boxes.cls.int().tolist(),
                                    r.boxes.conf.tolist()):
            x1, y1, x2, y2 = xyxy
            x1, y1, x2, y2 = int(x1 + ox), int(y1 + oy), int(x2 + ox), int(y2 + oy)
            if not filtra_area((x1, y1, x2, y2), W, H):
                continue  # caixa gigante -> lixo
            if tid not in locked:
                votes[tid][NAMES[c]] += cf
                seen[tid] += 1
                top, n = votes[tid].most_common(1)[0]
                if seen[tid] >= LOCK_MIN_FRAMES and n >= LOCK_RATIO * sum(votes[tid].values()):
                    locked[tid] = top
            cls = locked.get(tid)
            color = COLORS[cls] if cls else (160, 160, 160)
            label = f'#{tid} {cls}' if cls else f'#{tid} analisando…'
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, label, (x1, max(18, y1 - 6)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
    cv2.rectangle(frame, (int(ROI[0]*W), int(ROI[1]*H)), (int(ROI[2]*W), int(ROI[3]*H)),
                  (0, 255, 255), 1)
    writer.write(frame)
cap.release()
writer.release()

for tid, cnt in votes.items():
    locked.setdefault(tid, cnt.most_common(1)[0][0])
intact = sum(1 for c in locked.values() if c == 'intact')
print(f'grãos: {len(locked)} | intactos: {intact}')
!cp {OUT} /content/drive/MyDrive/
print('vídeo no Drive: video_boxes_filtrado.mp4')

## Checklist final
1. Etapa 1: escolheu a config vencedora (imgsz/iou)? Anote — vai pro `demo_servidor_colab.ipynb` e pro track.
2. Etapa 2: labels justos? Se não, o conserto é no BUILDER (me avise), não no treino.
3. Etapa 3: só rode se 1–2 não bastarem. Compare pelo **mAP50-95** e pelo vídeo.
4. Etapa 4: calibre `ROI` e `MAX_AREA_FRAC` e me diga os valores — integro no servidor e no auto-treino v4.